# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the **FAIR² Mental Health Survey Dataset from Kilifi County, Kenya** using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata
metadata = dataset.metadata  # Using metadata object!
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

### Croissant Entities
- **Record Sets**: Logical collections of records (tables/files)
- **Fields**: Schema-level fields defined using `@id`, `name`, `dataType`, etc.
- **Columns**: Associated with a data file or logical entity.

Below, we print the available record sets and fields with their `@id`.

In [ ]:
# List all record sets with their @id
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"@id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")

# Inspect the first record set's fields and columns
if dataset.record_sets:
    main_record_set_id = dataset.record_sets[0]['@id']
    print(f"\nFields in Record Set '@id': {main_record_set_id}")
    fields = dataset.fields(record_set=main_record_set_id)
    for field in fields:
        print(f"  Field @id: {field['@id']} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")

    # Print columns (if present)
    columns = dataset.columns(record_set=main_record_set_id)
    print(f"\nColumns in Record Set '@id': {main_record_set_id}")
    for column in columns:
        print(f"  Column @id: {column['@id']} | name: {column.get('name', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

All entities are referenced using their `@id` as per Croissant conventions.

For this dataset, we will use the primary record set.

In [ ]:
# Prepare to extract records from all available record sets
record_set_ids = [record_set['@id'] for record_set in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"\nLoaded DataFrame for record set @id: {record_set_id}, shape: {dataframes[record_set_id].shape}")
    print(f"Columns (@id): {dataframes[record_set_id].columns.tolist()}")
    print(dataframes[record_set_id].head())

# For further exploration, select the main record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    main_df = dataframes[main_rs_id]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes.

### Example: Analyze GAD-7 Scores
- Assume GAD-7 (anxiety) scores are stored in the column with `@id` `'https://api.app.sen.science/frontiers/7160411/gad7_score'`.
- We will filter, normalize, and group by education (`@id` `'https://api.app.sen.science/frontiers/7160411/level_of_education'`).

In [ ]:
# Define field @ids to use (replace these with correct @ids from the overview if needed)
gad7_score_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'  # GAD-7 score
education_id = 'https://api.app.sen.science/frontiers/7160411/level_of_education'  # education

# Ensure columns exist in loaded DataFrame
if gad7_score_id in main_df.columns:
    # Filter: keep records where GAD-7 score > threshold
    threshold = 10
    filtered_df = main_df[main_df[gad7_score_id] > threshold]
    print(f"Filtered records with GAD-7 score (@id={gad7_score_id}) > {threshold}:")
    print(filtered_df[[gad7_score_id]].head())

    # Normalize GAD-7 score
    filtered_df[f"{gad7_score_id}_normalized"] = (filtered_df[gad7_score_id] - filtered_df[gad7_score_id].mean()) / filtered_df[gad7_score_id].std()
    print(f"Normalized GAD-7 scores:")
    print(filtered_df[[gad7_score_id, f"{gad7_score_id}_normalized"]].head())

    # Group by education if field exists
    if education_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(education_id)[gad7_score_id].mean().reset_index()
        print(f"Mean GAD-7 score grouped by education (@id={education_id}):")
        print(grouped_df.head())

else:
    print(f"Column with @id {gad7_score_id} not found. Please use the Data Overview output to select correct @ids.")

## 5. Visualization
Visualize the distribution of GAD-7 scores and relationship to level of education.

- Histograms for GAD-7 scores
- Bar plot of mean GAD-7 by education level

All fields referenced using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of GAD-7 scores
if gad7_score_id in main_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[gad7_score_id].dropna(), bins=15, kde=True)
    plt.title('GAD-7 Score Distribution (@id={})'.format(gad7_score_id))
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    # Bar plot: mean GAD-7 per education level
    if education_id in main_df.columns:
        group_means = main_df.groupby(education_id)[gad7_score_id].mean()
        plt.figure(figsize=(10,6))
        group_means.plot(kind='bar')
        plt.title('Mean GAD-7 Score by Education Level (@id={})'.format(education_id))
        plt.xlabel('Level of Education (@id)')
        plt.ylabel('Mean GAD-7 Score')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey dataset using Croissant standards via the `mlcroissant` library.

- Data was accessed and analyzed exclusively referencing all entities by their `@id` values.
- Key exploratory steps included overview, filtering, normalization, grouping, and visualization.
- The Croissant metadata and schema provide robust provenance and interoperability for FAIR workflows.

For deeper analysis, continue referencing fields via their `@id` and consult the Croissant schema for additional record sets, fields, and documentation.